# Análise B1 — Drift estatístico (Colab)

Adaptador fino que monta o Google Drive, atualiza o repositório e chama `scripts/drift/analise_b1.py`. **Nenhuma lógica científica vive neste notebook** (CLAUDE.md §5).

Lê os artefatos produzidos por `run_b1_statistical_drift.ipynb` em `artifacts/drift/b1_statistical/`, seleciona a corrida principal por (granularidade, escopo) descartando smoke tests, e produz tabela Wanderley-style + figuras estilo Wanderley Figura 2 em `artifacts/drift/analises/b1/<timestamp>/`.

**Pré-requisitos:**
- Pelo menos uma corrida não-smoke de `run_b1.py` em `artifacts/drift/b1_statistical/` (típico: 6 combos, 2 granularidades × 3 escopos).
- Runtime → Change runtime type → **CPU basta** (a análise é leve; sem GPU).

**Saída** em `MyDrive/ptbr-market-classification/artifacts/drift/analises/b1/<timestamp>/`:
- `tabela_wanderley.csv` + `tabela_wanderley.md`: agregação por (escopo, granularidade, teste, condicao).
- `figura_serie_temporal_<escopo>_<granularidade>.{png,svg}`: p-value time-ordered por par-de-janelas, uma curva por teste.
- `figura_painel_<granularidade>.{png,svg}`: 3 escopos empilhados pra comparação visual.
- `analise.md`: resumo dos achados.
- `runs_selecionados.json`: trilha de auditoria das corridas escolhidas.

**Tempo esperado**: <1 min para todos os artefatos.

## 1. Parâmetros

In [ ]:
REPO_URL = 'https://github.com/almeidadm/ptbr-market-classification-2.git'  # substituir pela URL do seu fork
RAMO = 'main'

DIR_REPO = '/content/ptbr-market-classification'
DIR_DRIVE = '/content/drive/MyDrive/ptbr-market-classification'

RAIZ_B1 = f'{DIR_DRIVE}/artifacts/drift/b1_statistical'
DIR_ANALISES = f'{DIR_DRIVE}/artifacts/drift/analises/b1'

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / atualizar repositório

In [ ]:
import os, subprocess

if os.path.exists(DIR_REPO):
    subprocess.run(['git', '-C', DIR_REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'checkout', RAMO], check=True)
    subprocess.run(['git', '-C', DIR_REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', RAMO, REPO_URL, DIR_REPO], check=True)

os.chdir(DIR_REPO)
print('cwd =', os.getcwd())

## 4. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 5. Validar presença dos runs B1

In [ ]:
from pathlib import Path

raiz = Path(RAIZ_B1)
assert raiz.exists(), f'Raiz de B1 não encontrada em {raiz}. Rode run_b1.ipynb antes.'
runs = [d for d in sorted(raiz.iterdir()) if d.is_dir()]
print(f'{len(runs)} subdiretórios encontrados em {raiz}:')
for d in runs:
    parquet = d / 'results.parquet'
    n = '?' if not parquet.exists() else f'{parquet.stat().st_size / 1024:.1f} KB'
    print(f'  {d.name} ({n})')

## 6. Executar análise

In [ ]:
import subprocess
res = subprocess.run(
    ['python', 'scripts/drift/analise_b1.py',
     '--raiz-b1', RAIZ_B1,
     '--out', DIR_ANALISES],
    capture_output=True, text=True
)
print('EXIT:', res.returncode)
print('--- STDOUT ---')
print(res.stdout)
if res.stderr:
    print('--- STDERR ---')
    print(res.stderr)
assert res.returncode == 0, 'analise_b1.py falhou — veja STDERR acima.'

## 7. Exibir artefatos gerados

In [ ]:
from pathlib import Path
import re

dirs_analise = sorted(Path(DIR_ANALISES).iterdir(), key=lambda p: p.name)
dir_atual = dirs_analise[-1]
print(f'Última análise: {dir_atual}\n')
for p in sorted(dir_atual.iterdir()):
    print(f'  {p.name} ({p.stat().st_size:,} bytes)')

## 8. Tabela Wanderley + resumo (inline)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((dir_atual / 'analise.md').read_text()))

## 9. Figuras (inline)

In [ ]:
from IPython.display import Image, display
for figura in sorted(dir_atual.glob('figura_*.png')):
    print(figura.name)
    display(Image(filename=str(figura)))